In [21]:
print('Enterprise Knowledge Assistant with Advanced RAG')

Enterprise Knowledge Assistant with Advanced RAG


In [22]:
%pip install langchain-core==0.3.19 langchain-text-splitters==0.3.2 pypdf==6.1.2 python-docx beautifulsoup4 langchain-openai==0.2.7 langchain-google-genai==2.0.6 openai==1.55.3 faiss-cpu langchain_classic rank_bm25

%pip install -U langchain-community langchain-pinecone


Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_core-0.3.19-py3-none-any.whl.metadata (6.3 kB)
  Using cached langchain_text_splitters-0.3.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached langchain_openai-0.2.7-py3-none-any.whl.metadata (2.6 kB)
  Using cached openai-1.55.3-py3-none-any.whl.metadata (24 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of langchain-classic to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_classic-1.0.6-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_classic-1.0.5-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_classic-1.0.4-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_classic-1.0.2-py3

ERROR: Cannot install langchain-classic==1.0.0, langchain-classic==1.0.1, langchain-classic==1.0.2, langchain-classic==1.0.3, langchain-classic==1.0.4, langchain-classic==1.0.5, langchain-classic==1.0.6, langchain-classic==1.0.7, langchain-classic==1.0.8, langchain-core==0.3.19 and langchain-text-splitters==0.3.2 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [23]:
import sys
import rank_bm25
import langchain_core
import pypdf
import docx
import bs4
import langchain_text_splitters

print(sys.executable)
print('rank_bm25 loaded from:', rank_bm25.__file__)
print('Notebook dependencies imported successfully')

C:\Users\Vinay\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
rank_bm25 loaded from: C:\Users\Vinay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\rank_bm25.py
Notebook dependencies imported successfully


In [24]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

bm25_check = BM25Retriever.from_documents([
    Document(page_content='Hybrid and remote work are supported.')
])
print('BM25Retriever check passed:', type(bm25_check).__name__)

BM25Retriever check passed: BM25Retriever


In [25]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

from utils.loader import load_document
from utils.splitter import split_document

import warnings
warnings.filterwarnings('ignore')

In [26]:
import os
import hashlib
from utils.embedder import build_or_update_vectorstores
from langchain_core.documents import Document
import json

directory = './policy_documents'
all_chunks: list[Document] = []

for file in os.listdir(directory):
    file_path = os.path.join(directory, file)

    if not os.path.isfile(file_path):
        continue

    docs = load_document(file_path)
    print('Loaded documents', len(docs))
    print('First 50 chars: ', docs[0].page_content[:100])

    print('=====Splitting into chunks ===========')
    chunk = split_document(docs, 5000, 200)
    all_chunks.extend(chunk)

    print('Total chunks: ', len(chunk))
    print(f'Chunk 0 (first 200 chars): \n{chunk[0].page_content[:1000]}')
    print(f'chunk metadata {chunk[0].metadata}')

    with open('./config/config.json', 'r') as f:
        config = json.load(f)

    vectorstore = build_or_update_vectorstores(chunk, config)

type of the text <class 'str'>
type of the text <class 'str'>
type of the text <class 'str'>
Loaded documents 3
First 50 chars:  Verdant	Peak	Technologies	-
Employee	Policies
Verdant	Peak	Technologies	—
Employee	Policies
1.	Intro
=====Splitting into chunks ===========
Total chunks:  3
Chunk 0 (first 200 chars): 
Verdant	Peak	Technologies	-
Employee	Policies
Verdant	Peak	Technologies	—
Employee	Policies
1.	Introduction	and	Purpose
Verdant	Peak	Technologies	(“VPT”	or	“the	Company”)	is	committed	to
maintaining	a	fair,	safe,	and	productive	workplace	for	every	employee.
This	Employee	Policies	document	establishes	the	general	rules,
expectations,	and	standards	that	govern	the	employment	relationship
between	VPT	and	its	workforce.	It	applies	to	all	full-time,	part-time,
contract,	and	temporary	employees	across	all	VPT	oﬀices,	including
Austin,	Bengaluru,	and	Krakow.	These	policies	work	alongside	the	HR
Handbook,	Leave	Policy,	IT	Policies,	Travel	Policy,	Beneﬁts
Documentation,	and	Code	of	Cond

In [27]:
from utils.utility import get_embedding_model

for provider in ['openai', 'gemini']:
    try:
        print(f'==testing the {provider.upper()}==')
        model = get_embedding_model(provider)
        print('model loded: ', type(model))
    except Exception as e:
        print(e)

==testing the OPENAI==
Open API key loaded!!
model loded:  <class 'langchain_openai.embeddings.base.OpenAIEmbeddings'>
==testing the GEMINI==
Gemini API key loaded!!
model loded:  <class 'langchain_google_genai.embeddings.GoogleGenerativeAIEmbeddings'>


In [28]:
#Symentic search
query = "Does VPT support hybrid or remote work?"
results = vectorstore.similarity_search(query, k =5)

print(f'Query: {query}\n\n')

for i, r in enumerate(results, 1):
    print(f'{i}: {r.page_content[:1000]}\n\n')

Query: Does VPT support hybrid or remote work?


1: Standard	working	hours	at	VPT	are	9:00	AM	to	6:00	PM	local	time,
Monday	through	Friday,	with	a	one-hour	unpaid	lunch	break,	totaling
40	hours	per	week.	Departments	with	operational	needs	(such	as
Customer	Support	and	IT	Operations)	may	operate	on	shift	schedules,
which	will	be	communicated	in	writing	by	the	relevant	manager.
Employees	are	expected	to	record	their	attendance	daily	through	the
VPT	Time	&	Attendance	Portal.	Repeated	unexplained	absences	(three
or	more	in	a	rolling	30-day	period)	will	trigger	an	automatic
notiﬁcation	to	the	employee’s	manager	and	HR	Business	Partner	for
follow-up.
5.	Flexible	and	Remote	Work
VPT	operates	under	a	Hybrid	Work	Model.	Employees	are	expected	to
be	present	in	their	assigned	oﬀice	location	a	minimum	of	three	days
per	week	(Tuesday,	Wednesday,	and	Thursday	are	designated	as	Core
Collaboration	Days).	The	remaining	two	days	may	be	worked	remotely,
subject	to	manager	approval	and	role	suitability.	F

In [29]:
from utils.retriever import get_retriever
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

retr = get_retriever(vectorstore, config) #base

query = "Does VPT support hybrid or remote work?"

provider = config.get("llm_provider", "openai").lower()
temperature = config.get("llm_temperature", 0.3)
llm = ChatOpenAI(
            model=config.get("openai_model", "gpt-4o-mini"),
            temperature=temperature,
            streaming=True  # future-proof for streaming
        )

print(f"Query: {query}\n")

res = retr.invoke(query)

context = '\n\n'.join(f"[source: {d.metadata.get('source')}]\n{d.page_content}" for d in res)

prompt = ChatPromptTemplate.from_messages([("system",
"You are an enterprise policy assistant. Answer ONLY from the context below."
"If the answer is not in the context, say you don't know. Cite the source filename.\n\n"
"Context: \n {context}"),
("human", "{query}")])

messages = prompt.format_messages(context = context, query=query)
#resp1 = llm.invoke(messages)
#print('The response is: ', resp1)

for i, r in enumerate(res, 1):
    print(f'{i}: {r.page_content[:500]}\n\n')

USing base retriever(simaple similarity search)
Query: Does VPT support hybrid or remote work?

1: Standard	working	hours	at	VPT	are	9:00	AM	to	6:00	PM	local	time,
Monday	through	Friday,	with	a	one-hour	unpaid	lunch	break,	totaling
40	hours	per	week.	Departments	with	operational	needs	(such	as
Customer	Support	and	IT	Operations)	may	operate	on	shift	schedules,
which	will	be	communicated	in	writing	by	the	relevant	manager.
Employees	are	expected	to	record	their	attendance	daily	through	the
VPT	Time	&	Attendance	Portal.	Repeated	unexplained	absences	(three
or	more	in	a	rolling	30-day	period)	wi


2: Standard	working	hours	at	VPT	are	9:00	AM	to	6:00	PM	local	time,
Monday	through	Friday,	with	a	one-hour	unpaid	lunch	break,	totaling
40	hours	per	week.	Departments	with	operational	needs	(such	as
Customer	Support	and	IT	Operations)	may	operate	on	shift	schedules,
which	will	be	communicated	in	writing	by	the	relevant	manager.
Employees	are	expected	to	record	their	attendance	daily	through	the


In [30]:
from utils.hybridSearchAndReRanker import create_hybrid_retriever

hybrid_retriever = create_hybrid_retriever(retr, all_chunks)

query = "Does VPT support hybrid or remote work?"

provider = config.get("llm_provider", "openai").lower()
temperature = config.get("llm_temperature", 0.3)
llm = ChatOpenAI(
            model=config.get("openai_model", "gpt-4o-mini"),
            temperature=temperature,
            streaming=True  # future-proof for streaming
        )

print(f"Query: {query}\n")

res1 = hybrid_retriever.invoke(query)

context = '\n\n'.join(f"[source: {d.metadata.get('source')}]\n{d.page_content}" for d in res1)

prompt = ChatPromptTemplate.from_messages([("system",
"You are an enterprise policy assistant. Answer ONLY from the context below."
"If the answer is not in the context, say you don't know. Cite the source filename.\n\n"
"Context: \n {context}"),
("human", "{query}")])

messages = prompt.format_messages(context = context, query=query)
#resp = llm.invoke(messages)
#print('The response is: ', resp)

for i, r in enumerate(res1, 1):
   print(f'\n----Rank {i}----')
   print(f'Source: {r.metadata.get('source', 'N/A')}')
   print(f'Page: {r.metadata.get('page', '-')}')
   print(f'Content:\n{r.page_content}')
   print('-' * 70)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 989.37it/s]


Query: Does VPT support hybrid or remote work?


----Rank 1----
Source: d:\AI application\Capstone Project\policy_documents\08_company_faqs.pdf:page: 1
Page: -
Content:
Verdant	Peak	Technologies	-
Company	FAQs
Verdant	Peak	Technologies	—
Company	FAQs
General
Q:	What	are	VPT’s	standard	working	hours?
	
A:	Standard	hours
are	9:00	AM	to	6:00	PM	local	time,	Monday	through	Friday,	with	a
one-hour	unpaid	lunch	break,	totaling	40	hours	per	week.	Some
departments,	such	as	Customer	Support,	operate	on	shift	schedules
communicated	separately	by	their	managers.
Q:	Does	VPT	support	hybrid	or	remote	work?
	
A:	Yes.	VPT	operates
a	Hybrid	Work	Model	requiring	at	least	three	in-oﬀice	days	per	week
(Tuesday–Thursday	are	Core	Collaboration	Days).	Fully	remote
arrangements	can	be	requested	on	a	case-by-case	basis	and	require
joint	manager	and	HR	approval.
Q:	How	do	I	update	my	personal	details	(address,	emergency
contact,	bank	account)?
	
A:	All	personal	details	can	be	updated
directly	through	the	Employ